# AI Physics Tutor — Colab Runner (clean version)

MSc Thesis: **AI Assistant — A Multimodal Conversational Agent with Realistic Avatar for Physics Education**
Author: Kopalidis Thomas — MSc AI & DL, University of West Attica

## How to use
1. **Runtime → Change runtime type → T4 GPU**
2. Always run **Section 1 (Setup)** top to bottom in a new runtime.
3. Then run **only the section you need** (2 Smoke tests, 3 Benchmarks, 4 Gemini, 5 Avatar, 6 Live demo).
4. **Section 7** is troubleshooting — run only if something breaks.
5. Finish with **Section 8 (Stop)** so the runtime does not keep running.

Big files (models, PDFs, ChromaDB, results) live in Drive: `MyDrive/thesis_ai_and_dl`. Code comes from GitHub.

# 1. Setup (run every new runtime, top to bottom)

## Cell 1 — Verify runtime and GPU

In [1]:
import subprocess, sys, platform
print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print('')
print('=== GPU ===')
try:
    out = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True)
    print(out.strip())
except Exception as e:
    print(f'nvidia-smi failed: {e}  (are you on a GPU runtime?)')
print('')
print('=== Disk ===')
print(subprocess.check_output(['df', '-h', '/content'], text=True).strip())

Python: 3.13.15
Platform: Linux-6.6.122+-x86_64-with-glibc2.39

=== GPU ===
Tesla T4, 15360 MiB, 580.82.07

=== Disk ===
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   43G   71G  38% /


## Cell 2 — Mount Google Drive

Drive gives us persistent storage so Ollama model downloads and benchmark results survive runtime resets.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
THESIS_ROOT = '/content/drive/MyDrive/thesis_ai_and_dl'
os.makedirs(f'{THESIS_ROOT}/ollama_models', exist_ok=True)
os.makedirs(f'{THESIS_ROOT}/chroma_db', exist_ok=True)
os.makedirs(f'{THESIS_ROOT}/results', exist_ok=True)
os.makedirs(f'{THESIS_ROOT}/logs', exist_ok=True)
print(f'Thesis Drive folder: {THESIS_ROOT}')
!ls -la {THESIS_ROOT}

Mounted at /content/drive
Thesis Drive folder: /content/drive/MyDrive/thesis_ai_and_dl
total 4752
drwx------ 7 root root    4096 Oct 10 22:11 .
drwx------ 8 root root    4096 Oct 18 13:36 ..
drwx------ 2 root root    4096 Oct 10 23:05 chroma_db
-rw------- 1 root root 4836487 Oct 10 22:53 chunks.jsonl
drwx------ 2 root root    4096 Oct 10 22:11 logs
drwx------ 2 root root    4096 Oct 10 22:11 ollama_models
drwx------ 2 root root    4096 Oct 10 22:43 openstax_pdfs
drwx------ 2 root root    4096 Oct 10 22:11 results


## Cell 3 — Clone the project repository

Clones into `/content/thesis_ai_and_dl` (fresh each runtime). Change `BRANCH` when needed.

In [3]:
import os, shutil
PROJECT_DIR = '/content/thesis_ai_and_dl'
REPO = 'https://github.com/thomaskopalidis/thesis_ai_and_dl.git'
BRANCH = 'avatar-interface'   # back to 'master' after merge

if os.path.exists(PROJECT_DIR):
    print(f'Removing existing {PROJECT_DIR}')
    shutil.rmtree(PROJECT_DIR)

!git clone --depth 1 -b {BRANCH} {REPO} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git log --oneline -5

Cloning into '/content/thesis_ai_and_dl'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 71 (delta 12), reused 65 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 1.09 MiB | 32.90 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/thesis_ai_and_dl
6078922 (grafted, HEAD -> avatar-interface, origin/avatar-interface) Restore edge-tts 7.3.1 (fixes Microsoft 403)


## Cell 4 — Install Python dependencies

We pin versions known to work together (see docs/LAB_NOTEBOOK.md for the saga).
Colab usually has torch+cuda pre-installed; we skip reinstalling torch to avoid breaking the CUDA stack.

In [4]:
# Minimal install — only what Colab lacks. No version pins (use what resolves).
import subprocess, sys

packages_to_install = [
    "faster-whisper",
    "chromadb",
    "edge-tts",
    "pymupdf",
    "soxr",
    "einops",
]

print(f'Installing: {", ".join(packages_to_install)}')
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages_to_install,
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else '')
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

print('\n=== Verification ===')
import importlib
for mod in ['torch', 'numpy', 'cv2', 'librosa', 'sentence_transformers',
            'chromadb', 'fastapi', 'faster_whisper', 'edge_tts',
            'soundfile', 'scipy', 'pkg_resources', 'einops']:
    try:
        m = importlib.import_module(mod)
        v = getattr(m, '__version__', '?')
        print(f'OK   {mod:28s} {v}')
    except Exception as e:
        print(f'FAIL {mod}: {type(e).__name__}: {str(e)[:100]}')

Installing: faster-whisper, chromadb, edge-tts, pymupdf, soxr, einops
kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 16.0 MB/s eta 0:00:00


=== Verification ===
OK   torch                        2.11.0+cu130
OK   numpy                        2.1.3
OK   cv2                          4.14.0
OK   librosa                      0.11.0
OK   sentence_transformers        5.7.0
OK   chromadb                     1.5.9
OK   fastapi                      0.141.1
OK   faster_whisper               1.2.1
OK   edge_tts                     7.3.1
OK   soundfile                    0.

/usr/lib/python3.13/importlib/__init__.py:88: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _bootstrap._gcd_import(name[level:], package, level)


## Cell 6 — Install ffmpeg (needed by STT and Wav2Lip output encoding)

In [5]:
!apt-get -qq update && apt-get install -y -qq ffmpeg libgl1 libglib2.0-0 >/dev/null
!ffmpeg -version | head -1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


## Cell 6b — Install Ollama (official installer)

Record the printed version in LAB_NOTEBOOK: all final benchmarks must use the same version.

In [6]:
import subprocess, time, os
subprocess.run(['pkill', '-9', '-f', 'ollama'], check=False)
time.sleep(2)
!apt-get install -y -qq zstd pciutils lshw >/dev/null
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -3
os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')
!ollama --version

>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Cell 7 — Start Ollama server in background

Models are read from Drive (`OLLAMA_MODELS`), so they are downloaded only once.

In [7]:
import subprocess, time, requests, os
subprocess.run(['pkill', '-9', '-f', 'ollama'], check=False)
time.sleep(3)

env = os.environ.copy()
env['OLLAMA_MODELS'] = f'{THESIS_ROOT}/ollama_models'
env['PATH'] = '/usr/local/bin:' + env.get('PATH', '')
log_path = f'{THESIS_ROOT}/logs/ollama_serve.log'
with open(log_path, 'ab') as f:
    subprocess.Popen(['/usr/local/bin/ollama', 'serve'], stdout=f, stderr=subprocess.STDOUT, env=env)

for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=1)
        if r.ok:
            print(f'✓ Ollama ready after {i+1}s')
            print('Models:', [m['name'] for m in r.json().get('models', [])])
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print('[FAIL] Ollama did not start. Last log lines:')
    !tail -30 {log_path}

✓ Ollama ready after 18s
Models: ['qwen2.5:3b', 'llama3.2:3b', 'phi3:mini', 'mistral:7b', 'llama3.1:8b', 'qwen2.5:7b']


## Cell 8 — Pull the LLM models (only the first time; they stay in Drive)

In [8]:
MODELS_TO_PULL = [
    'qwen2.5:3b',      # 1.9 GB
    'qwen2.5:7b',      # 4.7 GB
    'llama3.1:8b',     # 4.9 GB
    'llama3.2:3b',     # 2.0 GB
    'mistral:7b',      # 4.1 GB
    'phi3:mini',       # 2.2 GB
    #'gemma3:1b',      # smoke tests only
    #'qwen2.5:14b-instruct-q4_K_M',  # only if GPU has >= 20 GB VRAM
]
for m in MODELS_TO_PULL:
    print(f'\n=== Pulling {m} ===')
    !ollama pull {m} 2>&1 | tail -2
print('\n=== Final list ===')
!ollama list


=== Pulling qwen2.5:3b ===
removing unused layers 
success 

=== Pulling qwen2.5:7b ===
writing manifest 
success 

=== Pulling llama3.1:8b ===
writing manifest 
success 

=== Pulling llama3.2:3b ===
writing manifest 
success 

=== Pulling mistral:7b ===
writing manifest 
success 

=== Pulling phi3:mini ===
writing manifest 
success 

=== Final list ===
NAME           ID              SIZE      MODIFIED      
phi3:mini      4f2222927938    2.2 GB    1 second ago     
mistral:7b     6577803aa9a0    4.4 GB    2 seconds ago    
llama3.2:3b    a80c4f17acd5    2.0 GB    3 seconds ago    
llama3.1:8b    46e0c10c039e    4.9 GB    4 seconds ago    
qwen2.5:7b     845dbda0ea48    4.7 GB    5 seconds ago    
qwen2.5:3b     357c53fb659c    1.9 GB    6 seconds ago    


## Cell 9 — Download OpenStax PDFs and build the KB index

The PDFs are not in git (too large). We download them once to Drive, run ingestion to produce `chunks.jsonl`, then build the ChromaDB index. All outputs live in Drive — next runtime restart will just reuse them.

In [9]:
import os, urllib.request

KB_DIR = f'{PROJECT_DIR}/Knowledge_base'
SRC_DIR = f'{KB_DIR}/sources'
DRIVE_PDFS = f'{THESIS_ROOT}/openstax_pdfs'
os.makedirs(DRIVE_PDFS, exist_ok=True)
os.makedirs(SRC_DIR, exist_ok=True)

OPENSTAX = {
    'openstax_vol1.pdf': 'https://assets.openstax.org/oscms-prodcms/media/documents/UniversityPhysicsVolume1-WEB_Zwi59QO.pdf',
    'openstax_vol2.pdf': 'https://assets.openstax.org/oscms-prodcms/media/documents/UniversityPhysicsVolume2-WEB_oqeqFz9.pdf',
    'openstax_vol3.pdf': 'https://assets.openstax.org/oscms-prodcms/media/documents/UniversityPhysicsVolume3-WEB_8rXeDQZ.pdf',
}

for fname, url in OPENSTAX.items():
    drive_path = f'{DRIVE_PDFS}/{fname}'
    project_path = f'{SRC_DIR}/{fname}'
    if not os.path.isfile(drive_path):
        print(f'Downloading {fname}...')
        urllib.request.urlretrieve(url, drive_path)
    # Symlink from project dir into Drive copy — no duplication
    if not os.path.islink(project_path) and not os.path.isfile(project_path):
        os.symlink(drive_path, project_path)
    print(f'OK  {fname} ({os.path.getsize(drive_path)/1e6:.1f} MB)')

OK  openstax_vol1.pdf (81.5 MB)
OK  openstax_vol2.pdf (63.8 MB)
OK  openstax_vol3.pdf (53.5 MB)


In [10]:
# If a previously built chunks.jsonl exists in Drive, reuse it; otherwise re-ingest.
import os
DRIVE_CHUNKS = f'{THESIS_ROOT}/chunks.jsonl'
PROJECT_CHUNKS = f'{PROJECT_DIR}/Knowledge_base/processed/chunks.jsonl'
os.makedirs(f'{PROJECT_DIR}/Knowledge_base/processed', exist_ok=True)

if os.path.isfile(DRIVE_CHUNKS):
    !cp {DRIVE_CHUNKS} {PROJECT_CHUNKS}
    print(f'Reused chunks.jsonl from Drive ({os.path.getsize(PROJECT_CHUNKS)/1e6:.1f} MB)')
else:
    !cd {PROJECT_DIR} && python evaluation/ingest_kb.py --chunk-size 220 --overlap 32 --preview 2
    !cp {PROJECT_CHUNKS} {DRIVE_CHUNKS}
    print('Fresh ingestion → saved to Drive for reuse')

Reused chunks.jsonl from Drive (4.8 MB)


In [11]:
# Build (or reuse) ChromaDB index. Store the DB in Drive so it persists.
import os
DRIVE_CHROMA = f'{THESIS_ROOT}/chroma_db'
PROJECT_CHROMA = f'{PROJECT_DIR}/chroma_db'

# Symlink project's chroma_db -> Drive's chroma_db so build_index writes to Drive directly
if os.path.exists(PROJECT_CHROMA) and not os.path.islink(PROJECT_CHROMA):
    import shutil; shutil.rmtree(PROJECT_CHROMA)
if not os.path.islink(PROJECT_CHROMA):
    os.symlink(DRIVE_CHROMA, PROJECT_CHROMA)

# Only build if the collection does not exist yet
import glob
if not glob.glob(f'{DRIVE_CHROMA}/*.sqlite3'):
    !cd {PROJECT_DIR} && python evaluation/build_index.py
else:
    print('ChromaDB index already present in Drive — skipping rebuild')
    !ls -la {DRIVE_CHROMA}

ChromaDB index already present in Drive — skipping rebuild
total 35856
drwx------ 3 root root     4096 Oct 10 23:05 .
drwx------ 7 root root     4096 Oct 10 22:11 ..
drwx------ 2 root root     4096 Oct 10 23:05 129b2ca1-0750-4ff3-8441-86e23fb87752
-rw------- 1 root root 36704256 Oct 16 08:29 chroma.sqlite3


## Cell 10 — Smoke test the RAG

In [12]:
import sys
sys.path.insert(0, PROJECT_DIR)
from core.rag import retrieve_with_metadata

r = retrieve_with_metadata("What is Newton's second law?", top_k=3)
for i, (cid, meta, dist, doc) in enumerate(zip(r['ids'], r['metadatas'], r['distances'], r['documents']), 1):
    print(f'[{i}] dist={dist:.3f}  {meta.get("chapter")}  /  {meta.get("section")}')
    print(f'    {doc[:140]}...')
    print()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[RAG] Loaded existing collection: 3950 chunks
[1] dist=0.385  Chapter 5 Newton's Laws of Motion  /  5.5 Newton’s Third Law
    Significance
To apply Newton’s second law as the primary equation in solving a problem, we sometimes have to rely on other
equations, such a...

[2] dist=0.410  Chapter 5 Newton's Laws of Motion  /  5.5 Newton’s Third Law
    When you push on a wall, the wall pushes back on you. This brings us to Newton’s third law. Newton’s third law represents a certain symmetry...

[3] dist=0.419  Chapter 5 Newton's Laws of Motion  /  5.3 Newton's Second Law
    If the magnitude of the truck’s acceleration is
 what is the magnitude of the sports
car’s acceleration? Component Form of Newton’s Second L...



##CELL 11 — Wav2Lip setup (για avatar)

In [13]:
import os
WAV2LIP_DIR = f'{PROJECT_DIR}/Wav2Lip'

if not os.path.exists(WAV2LIP_DIR):
    !cd {PROJECT_DIR} && git clone https://github.com/Rudrabha/Wav2Lip.git

# Patches
!sed -i 's|librosa.output.write_wav(path, wav, sr=sr)|import soundfile as _sf; _sf.write(path, wav, sr)|g' {WAV2LIP_DIR}/audio.py
!sed -i 's|librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,|librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft, n_mels=hp.num_mels,|g' {WAV2LIP_DIR}/audio.py

# Checkpoints
CKPT = f'{WAV2LIP_DIR}/checkpoints/wav2lip_gan.pth'
if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 400_000_000:
    os.makedirs(os.path.dirname(CKPT), exist_ok=True)
    !wget -q -O {CKPT} 'https://huggingface.co/numz/wav2lip_studio/resolve/main/Wav2lip/wav2lip_gan.pth'

FACE_DET = f'{WAV2LIP_DIR}/face_detection/detection/sfd/s3fd.pth'
if not os.path.exists(FACE_DET):
    os.makedirs(os.path.dirname(FACE_DET), exist_ok=True)
    !wget -q -O {FACE_DET} 'https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth'

!pip install -q batch-face face-alignment
print(f'\n✓ Wav2Lip: {os.path.getsize(CKPT)/1e6:.0f} MB')
print(f'✓ FaceDet: {os.path.getsize(FACE_DET)/1e6:.0f} MB')


Cloning into 'Wav2Lip'...
remote: Enumerating objects: 409, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 409 (delta 2), reused 2 (delta 2), pack-reused 405 (from 2)
Receiving objects: 100% (409/409), 549.05 KiB | 2.60 MiB/s, done.
Resolving deltas: 100% (227/227), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.6/30.6 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 3.8 MB/s eta 0:00:00

✓ Wav2Lip: 436 MB
✓ FaceDet: 90 MB


## Cell 18.5 — Re-create symlink + regenerate wavs

In [15]:
# Fix 1: Symlink chroma_db to Drive copy
import os, shutil

DRIVE_CHROMA = f'{THESIS_ROOT}/chroma_db'
PROJECT_CHROMA = f'{PROJECT_DIR}/chroma_db'

if os.path.exists(PROJECT_CHROMA) and not os.path.islink(PROJECT_CHROMA):
    shutil.rmtree(PROJECT_CHROMA)
if not os.path.islink(PROJECT_CHROMA):
    os.symlink(DRIVE_CHROMA, PROJECT_CHROMA)
print(f'✓ Symlink: {PROJECT_CHROMA} → {DRIVE_CHROMA}')
!ls -lah {DRIVE_CHROMA}

# Fix 2: Regenerate synthetic wavs (δεν υπάρχουν στο GitHub)
import subprocess
os.chdir(PROJECT_DIR)
os.makedirs(f'{PROJECT_DIR}/evaluation/audio/synthetic', exist_ok=True)
print('\n=== Generating synthetic wavs ===')
result = subprocess.run(
    ['python', 'evaluation/generate_wer_audio.py'],
    capture_output=True, text=True, timeout=180
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
!ls {PROJECT_DIR}/evaluation/audio/synthetic/


✓ Symlink: /content/thesis_ai_and_dl/chroma_db → /content/drive/MyDrive/thesis_ai_and_dl/chroma_db
total 36M
drwx------ 3 root root 4.0K Oct 18 13:38 .
drwx------ 7 root root 4.0K Oct 10 22:11 ..
drwx------ 2 root root 4.0K Oct 10 23:05 129b2ca1-0750-4ff3-8441-86e23fb87752
-rw------- 1 root root  36M Oct 18 13:38 chroma.sqlite3

=== Generating synthetic wavs ===
[TTS] Q01  'What is the difference between distance and displacement?'
[TTS] Q03  'A ball is thrown horizontally from a cliff. Why does its hor…'
[TTS] Q05  'State Newton's first law of motion.'
[TTS] Q07  'If I push a wall, Newton's third law says the wall pushes ba…'
[TTS] Q10  'State the law of conservation of mechanical energy and when …'
[TTS] Q13  'State Archimedes' principle.'
[TTS] Q16  'What is the difference between temperature and heat?'
[TTS] Q22  'Why do resistors in series add directly while resistors in p…'
[TTS] Q24  'State Snell's law of refraction.'
[TTS] Q28  'What is time dilation in special relativity?'

[O

# 2. Smoke tests

## Cell 11 — Smoke test the LLM

One question through qwen2.5:3b to confirm Ollama + LLM wrapper work on GPU.

In [ ]:
import time, os, requests, sys

MODEL = 'qwen2.5:3b'
os.environ['OLLAMA_MODEL'] = MODEL
os.environ['OLLAMA_READ_TIMEOUT'] = '300'  # 5 min for cold loads from Drive

# Clear cached python modules so new env vars take effect
for mod in list(sys.modules.keys()):
    if mod == 'config' or mod.startswith('core.'):
        del sys.modules[mod]
sys.path.insert(0, PROJECT_DIR)
import config
from core.llm import stream_llm_response

# --- Warm-up: ensure the model is loaded into VRAM before measuring ---
print(f'[WARMUP] Loading {MODEL} into VRAM (may take up to 60s on first call)...')
t_warm = time.time()
try:
    r = requests.post(
        'http://localhost:11434/api/generate',
        json={'model': MODEL, 'prompt': 'hi', 'stream': False,
              'options': {'num_predict': 1}, 'keep_alive': '30m'},
        timeout=(10, 300),
    )
    print(f'[WARMUP] model ready in {time.time()-t_warm:.1f}s (status {r.status_code})')
except Exception as e:
    print(f'[WARMUP] FAILED: {e}')
    raise

# --- Measured query ---
query = "What is Newton's second law?"
print(f'\n[QUERY] {query}')

t0 = time.time()
first_token_t = None
parts = []
for token in stream_llm_response(query, model=MODEL):
    if first_token_t is None:
        first_token_t = time.time() - t0
        print(f'[TIMING] first token: {first_token_t:.2f}s')
    parts.append(token)

answer = ''.join(parts).strip()
total = time.time() - t0
print(f'[TIMING] total: {total:.2f}s  ({len(parts)} tokens, {len(answer)} chars)')
print(f'\nA: {answer}')


## Cell 12 — Smoke test TTS

Synthesise one short sentence to confirm edge-tts works in the sandbox.

In [ ]:
import asyncio, time
from core.tts import text_to_speech_stream

async def _test():
    t0 = time.time()
    nbytes = 0
    async for chunk in text_to_speech_stream('Hello from Google Colab.'):
        nbytes += len(chunk)
    print(f'[TIMING] TTS: {time.time()-t0:.2f}s  ({nbytes} bytes of mp3 audio)')

await _test()

## Cell 13 — Status summary

Everything above should have run without errors. If so, we are ready for the benchmark script (next step).

In [ ]:
import subprocess
print('=== Status ===')
print(f'Project:   {PROJECT_DIR}')
print(f'Drive:     {THESIS_ROOT}')
!which ollama
print('')
print('=== Loaded models (VRAM) ===')
!ollama ps
print('')
print('=== GPU utilisation ===')
!nvidia-smi --query-gpu=name,utilization.gpu,memory.used,memory.total --format=csv,noheader

# 3. Benchmarks

## 3a — LLM + RAG only (`evaluation/benchmark.py`)
First a quick check (2 questions), then the full run.

In [ ]:
import os
os.chdir(PROJECT_DIR)
os.environ['OLLAMA_READ_TIMEOUT'] = '300'

!python evaluation/benchmark.py \
    --models qwen2.5:3b \
    --rag both \
    --runs 1 \
    --warmups 1 \
    --limit-questions 2


In [ ]:
import os
os.chdir(PROJECT_DIR)
os.environ['OLLAMA_READ_TIMEOUT'] = '300'

!python evaluation/benchmark.py \
    --models qwen2.5:3b qwen2.5:7b llama3.1:8b \
    --rag both \
    --runs 3 \
    --warmups 1


## 3b — Full pipeline (`evaluation/benchmark_full_pipeline.py`)
Smoke test (1 model, 2 questions), then all 6 models.

In [ ]:
import os
os.chdir(PROJECT_DIR)
os.environ['OLLAMA_READ_TIMEOUT'] = '300'

!python evaluation/benchmark_full_pipeline.py \
    --models qwen2.5:3b \
    --runs 1 --warmups 1 \
    --limit-questions 2


## Cell 20 Full Benchmark (6 models)
1. qwen2.5:3b
2. qwen2.5:7b
3. llama3.1:8b
4. mistral:7b
5. phi3:mini
6. llama3.2:3b

In [ ]:
import os
os.chdir(PROJECT_DIR)
os.environ['OLLAMA_READ_TIMEOUT'] = '300'

!python evaluation/benchmark_full_pipeline.py \
    --models qwen2.5:3b qwen2.5:7b llama3.1:8b mistral:7b phi3:mini llama3.2:3b \
    --runs 1 --warmups 1

## 3c — Results table (300 dpi)
Change `summary_file` to the CSV you want to plot.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

summary_file = f'{PROJECT_DIR}/evaluation/results/stages_summary_20261011_104315.csv'
df = pd.read_csv(summary_file)

# Prepare display
df = df[['model', 'stt_s', 'rag_s', 'llm_s', 'tts_s', 'avatar_s', 'total_s']]
df = df.round({'stt_s': 3, 'rag_s': 3, 'llm_s': 2, 'tts_s': 2, 'avatar_s': 2, 'total_s': 2})
df = df.sort_values('total_s').reset_index(drop=True)

# Rename columns for publication
df.columns = ['Model', 'STT (s)', 'RAG (s)', 'LLM (s)', 'TTS (s)', 'Avatar (s)', 'Total (s)']

# Render table as image
fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')
tbl = ax.table(cellText=df.values, colLabels=df.columns,
               loc='center', cellLoc='center',
               colColours=['#4a90e2']*len(df.columns))

# Styling
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.1, 2.0)

# Make header white text
for i in range(len(df.columns)):
    tbl[(0, i)].set_text_props(weight='bold', color='white')

# Alternate row colors
for i in range(1, len(df)+1):
    for j in range(len(df.columns)):
        if i % 2 == 0:
            tbl[(i, j)].set_facecolor('#f0f0f0')

plt.title('Full Pipeline Benchmark — 6 LLMs on NVIDIA T4 (30 questions, RAG=on)',
          fontsize=13, fontweight='bold', pad=15)
plt.savefig('/content/stages_table.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('\n✓ Saved to /content/stages_table.png')

# 4. Cloud LLM (Gemini)

The key lives in Colab **Secrets** (key icon on the left) as `GEMINI_API_KEY`. Never print the key.

In [ ]:
from google.colab import userdata
key = userdata.get('GEMINI_API_KEY').strip()
print(f'Key loaded: length={len(key)}, valid prefix={key.startswith(("AIza", "AQ."))}')

## Cell 22 (quick Gemini API test)

In [ ]:
from google.colab import userdata
import requests

key = userdata.get('GEMINI_API_KEY').strip()

# Try newer models
for model in ['gemini-3.8-flash', 'gemini-3.8-pro', 'gemini-3.5-flash']:
    url = f'https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={key}'
    try:
        r = requests.post(url, json={
            'contents': [{'parts': [{'text': 'Say hello in one word.'}]}]
        }, timeout=30)
        if r.ok:
            print(f'✓ {model}: {r.json()["candidates"][0]["content"]["parts"][0]["text"]}')
        else:
            err = r.json().get('error', {}).get('message', r.text[:200])
            print(f'✗ {model}: {r.status_code} - {err[:150]}')
    except Exception as e:
        print(f'✗ {model}: {e}')


## Cell 24 Quick Gemini test μέσω του project wrapper

In [ ]:
import os, sys
from google.colab import userdata

# Set env var so core/llm.py picks it up
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

# Clear module cache for fresh config
for mod in list(sys.modules.keys()):
    if mod == 'config' or mod.startswith('core.'):
        del sys.modules[mod]

sys.path.insert(0, PROJECT_DIR)
from core.llm import stream_llm_response
import time

t0 = time.time()
parts = []
for tok in stream_llm_response("What is Newton's second law?", model='cloud:google:gemini-3.5-flash'):
    parts.append(tok)
answer = ''.join(parts).strip()
print(f'[TIMING] {time.time()-t0:.2f}s')
print(f'A: {answer}')


## Cell 25 - Benchmark Gemini

In [ ]:
import os
os.chdir(PROJECT_DIR)
os.environ['OLLAMA_READ_TIMEOUT'] = '300'
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

!python evaluation/benchmark_full_pipeline.py \
    --models cloud:google:gemini-3.5-flash \
    --runs 1 --warmups 1 \
    --skip-stt --skip-avatar


# 5. Avatar experiments (Wav2Lip / SadTalker / MuseTalk)

Uses the common `AvatarEngine` interface (`core/avatar_engine.py`). Needs Section 1 (Wav2Lip setup + wavs). Ollama is not needed here (D-011: stop it during avatar measurements).

## 5.0 — Test 3b: Wav2Lip on a VIDEO base

In [16]:
import os, sys
os.chdir(PROJECT_DIR); sys.path.insert(0, PROJECT_DIR)
for m in list(sys.modules):                      # fresh imports
    if m == 'config' or m.startswith('core'):
        del sys.modules[m]
from core import avatar_engine
print('Available engines:', avatar_engine.available_engines())

AUDIO = 'evaluation/audio/synthetic/Q05.wav'
assert os.path.isfile(AUDIO), 'Run the "regenerate wavs" cell in Section 1 first'

# 3-second test video, 25 fps, aspect ratio preserved
!ffmpeg -y -loglevel error -loop 1 -i frontend/assets/teacher_avatar.jpg -t 3 -r 25 -vf "scale=-2:512" -pix_fmt yuv420p -c:v libx264 _test_video.mp4

r = avatar_engine.get('wav2lip').generate(AUDIO, base_path='_test_video.mp4')
print(f'base_kind={r.base_kind} time={r.time_s:.2f}s dur={r.video_duration_s:.2f}s '
      f'fps={r.fps:.1f} vram={r.peak_vram_mb:.0f}MB encode={r.encode_time_s:.2f}s')

from IPython.display import Video
Video(r.video_path, embed=True, width=320)

Available engines: ['wav2lip']
[Wav2Lip] Prepared video: /content/thesis_ai_and_dl/_test_video.mp4
base_kind=video time=52.34s dur=2.76s fps=25.0 vram=10096MB encode=0.22s


## 5.1 — Idle video with SadTalker (STEP 1, to be added)

## 5.2 — MuseTalk (STEP 2, to be added)

# 6. Live demo

##CELL 12 — LIVE DEMO

In [ ]:
import subprocess, os, time, requests
from google.colab import userdata
from google.colab.output import eval_js

# ─── Kill previous server ───
subprocess.run(['pkill', '-9', '-f', 'app.py'], check=False)
subprocess.run(['pkill', '-9', '-f', 'uvicorn'], check=False)
time.sleep(2)

# ─── Load cloud API keys from secrets ───
for name in ['GEMINI_API_KEY', 'OPENROUTER_API_KEY', 'GROQ_API_KEY',
             'OPENAI_API_KEY', 'ANTHROPIC_API_KEY']:
    try:
        os.environ[name] = userdata.get(name)
        print(f'✓ {name}')
    except Exception:
        print(f'  (skip {name})')

# ─── Launch app.py ───
os.chdir(PROJECT_DIR)
print('\n🚀 Launching app.py...')
log_path = f'{THESIS_ROOT}/logs/app_server.log'
with open(log_path, 'wb') as log:
    proc = subprocess.Popen(['python', 'app.py'], stdout=log, stderr=subprocess.STDOUT)

# ─── Wait for server ready ───
ready = False
for i in range(60):
    try:
        r = requests.get('http://localhost:8000/api/models', timeout=1)
        if r.ok:
            print(f'\n✓ Server ready after {i+1}s')
            print(f'  Models available: {r.json()["models"]}')
            ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not ready:
    print('\n[FAIL] Server did not start. Logs:')
    !tail -30 {log_path}
else:
    # ─── Get public URL ───
    url = eval_js('google.colab.kernel.proxyPort(8000)')
    full_url = f'{url}app/'

    print('\n' + '═' * 70)
    print('🌐 LIVE DEMO URL (κάνε click για να ανοίξει σε νέο tab):')
    print('   ' + full_url)
    print('═' * 70)

    print('\n📝 Οδηγίες:')
    print('  1. Κάνε click στο URL παραπάνω')
    print('  2. Model dropdown:')
    print('     • "cloud:groq:qwen/qwen3.8-27b"  → fast (0.15s LLM)')
    print('     • "qwen2.5:3b"                    → local, για avatar')
    print('  3. Avatar toggle: OFF για γρήγορο | ON για video')
    print('  4. Chat: γράψε "What is Newton\'s second law?" ή οτιδήποτε')
    print('\n📊 Logs live:  !tail -f ' + log_path)
    print('🛑 Stop:        τρέξε το επόμενο cell')

    # Clickable HTML link
    from IPython.display import HTML, display
    display(HTML(f'<h2>🎬 <a href="{full_url}" target="_blank" style="color:#4a90e2;">ΑΝΟΙΞΕ ΤΟ LIVE DEMO ΕΔΩ ➡</a></h2>'))


### Alternative public URL via ngrok (needs `NGROK_TOKEN` in Secrets). Do not share the link.

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok, conf
from google.colab import userdata

conf.get_default().auth_token = userdata.get('NGROK_TOKEN')
ngrok.kill()

public_url = ngrok.connect(8000, "http")
demo_url = f"{public_url.public_url}/app/"
print(f"\n{'='*60}")
print(f"🎬 LIVE DEMO URL:")
print(f"{demo_url}")
print(f"{'='*60}\n")


### Restart the app after a `git pull`

In [ ]:
%cd /content/thesis_ai_and_dl
!git pull
!pkill -f "python3 app.py"
import time, subprocess
time.sleep(3)
subprocess.Popen(['python3', 'app.py'],
                 stdout=open('/tmp/app.log','w'),
                 stderr=subprocess.STDOUT,
                 cwd='/content/thesis_ai_and_dl')
time.sleep(6)
print("✓ Server restarted")
!tail -10 /tmp/app.log


##CELL 13 — Stop server

In [ ]:
import subprocess
subprocess.run(['pkill', '-9', '-f', 'app.py'], check=False)
subprocess.run(['pkill', '-9', '-f', 'uvicorn'], check=False)
print('✓ Server stopped')


# 7. Troubleshooting (run ONLY when something is broken)

### Is Ollama alive? Which models?

In [ ]:
import os, requests
print('=== Variables ===')
try:
    print(f'PROJECT_DIR: {PROJECT_DIR}')
    print(f'THESIS_ROOT: {THESIS_ROOT}')
except NameError as e:
    print(f'FAIL: {e}')

print('\n=== Ollama ===')
try:
    r = requests.get('http://localhost:11434/api/tags', timeout=3)
    print(f'OK, models: {[m["name"] for m in r.json().get("models", [])]}')
except Exception as e:
    print(f'DEAD: {e}')


### Ollama logs + GPU state

In [ ]:
# Get Ollama server logs
!tail -60 {THESIS_ROOT}/logs/ollama_serve.log

print('\n\n=== GPU state ===')
!nvidia-smi

print('\n=== CUDA version ===')
!nvcc --version 2>/dev/null || echo "nvcc not installed"

print('\n=== Ollama version ===')
!ollama --version


### Models loaded in VRAM

In [ ]:
import subprocess, requests

# Check if any model is loaded
print('=== Models loaded in GPU/CPU ===')
try:
    r = requests.get('http://localhost:11434/api/ps', timeout=3)
    print(r.json())
except Exception as e:
    print(f'FAIL: {e}')

print('\n=== Ollama logs (last 40) ===')
!tail -40 {THESIS_ROOT}/logs/ollama_serve.log

print('\n=== GPU state ===')
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader


### Python processes (app / ollama)

In [ ]:
!curl -s http://localhost:11434/api/tags | python3 -c "import json,sys; d=json.load(sys.stdin); print('Models loaded:', [m['name'] for m in d.get('models', [])])"
!ps aux | grep -E "ollama|python3 app" | grep -v grep


### App log

In [ ]:
!tail -50 /tmp/app.log


### OPTIONAL: copy Ollama models from Drive to local SSD (faster first load)

In [ ]:
import os, shutil, subprocess, time, requests

# Stop ollama first
subprocess.run(['pkill', '-9', '-f', 'ollama'], check=False)
time.sleep(3)

# Copy models from Drive to local SSD
LOCAL_MODELS = '/content/ollama_models'
DRIVE_MODELS = f'{THESIS_ROOT}/ollama_models'

if not os.path.exists(LOCAL_MODELS):
    print(f'Copying {DRIVE_MODELS} → {LOCAL_MODELS}')
    print('(takes ~2-3 min for 11GB)')
    shutil.copytree(DRIVE_MODELS, LOCAL_MODELS)
print('✓ Done')
!du -sh {LOCAL_MODELS}

# Restart Ollama pointing to local models
env = os.environ.copy()
env['OLLAMA_MODELS'] = LOCAL_MODELS
log_path = f'{THESIS_ROOT}/logs/ollama_serve.log'

with open(log_path, 'ab') as f:
    subprocess.Popen(['ollama', 'serve'], stdout=f, stderr=subprocess.STDOUT, env=env)

for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=1)
        if r.ok:
            print(f'✓ Ollama ready after {i+1}s')
            print('Models:', [m['name'] for m in r.json().get('models', [])])
            break
    except Exception:
        pass
    time.sleep(1)


### OPTIONAL: FORCE rebuild of the ChromaDB index (deletes and rebuilds it in Drive!)

In [ ]:
import os, shutil, sys, time

DRIVE_CHROMA = f'{THESIS_ROOT}/chroma_db'
PROJECT_CHROMA = f'{PROJECT_DIR}/chroma_db'

# 1. Αφαίρεσε symlink
if os.path.islink(PROJECT_CHROMA):
    os.unlink(PROJECT_CHROMA)

# 2. Nuke completely with sleep για να αφήσει time το Drive να sync
if os.path.exists(DRIVE_CHROMA):
    shutil.rmtree(DRIVE_CHROMA)
    print('✓ Nuked Drive chroma')
time.sleep(3)

# 3. Verify διαγράφηκε
print('Still exists?', os.path.exists(DRIVE_CHROMA))

# 4. Recreate
os.makedirs(DRIVE_CHROMA)
os.symlink(DRIVE_CHROMA, PROJECT_CHROMA)
print('✓ Fresh dirs')

# 5. Clear any Python-level chromadb caches
import importlib, sys
for mod in list(sys.modules.keys()):
    if 'chromadb' in mod or 'core.rag' in mod or mod == 'config':
        del sys.modules[mod]
print('✓ Cleared module cache')

# 6. Re-import fresh
sys.path.insert(0, PROJECT_DIR)
import config
import chromadb
from chromadb.utils import embedding_functions
import json

ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=config.RAG_EMBEDDING_MODEL
)
client = chromadb.PersistentClient(path=DRIVE_CHROMA)

# 7. Try delete first (idempotent)
try:
    client.delete_collection(config.CHROMA_COLLECTION_NAME)
    print(f'✓ Deleted existing collection')
except Exception as e:
    print(f'  (no existing collection to delete: {e})')

# 8. Create fresh with get_or_create as fallback
try:
    col = client.create_collection(config.CHROMA_COLLECTION_NAME, embedding_function=ef)
    print('✓ Created new collection')
except Exception as e:
    print(f'  create_collection failed: {e}')
    print('  Trying get_or_create...')
    col = client.get_or_create_collection(config.CHROMA_COLLECTION_NAME, embedding_function=ef)

# 9. Verify empty
print(f'Collection count before indexing: {col.count()}')

# 10. Load and index
CHUNKS_PATH = f'{PROJECT_DIR}/Knowledge_base/processed/chunks.jsonl'
documents, ids, metadatas = [], [], []
with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        c = json.loads(line)
        documents.append(c['text'])
        ids.append(c['chunk_id'])
        metadatas.append({
            'source': c['source'],
            'volume': c['volume'],
            'chapter': c['chapter'],
            'section': c['section'],
            'page_start': c['page_start'],
            'page_end': c['page_end'],
            'license': c['license'],
            'n_tokens': c['n_tokens'],
        })

print(f'\n✓ Loaded {len(documents)} chunks, indexing...')
BATCH = 256
for i in range(0, len(documents), BATCH):
    col.add(
        documents=documents[i:i+BATCH],
        ids=ids[i:i+BATCH],
        metadatas=metadatas[i:i+BATCH],
    )
    if (i // BATCH) % 4 == 0:
        print(f'[INDEX] {min(i+BATCH, len(documents))}/{len(documents)}')

print(f"\n[DONE] Collection '{config.CHROMA_COLLECTION_NAME}' now has {col.count()} chunks.")

# 11. CRITICAL verify - metadata exists
r = col.query(query_texts=["Newton second law"], n_results=1, include=["metadatas", "documents", "distances"])
print(f'\n=== VERIFY ===')
print(f'id: {r["ids"][0][0]}')
print(f'metadata: {r["metadatas"][0][0]}')


# 8. Stop the runtime (always at the end)

In [17]:
# === STOP NOW: save Drive and delete the runtime ===
from google.colab import runtime, drive
try: drive.flush_and_unmount()
except Exception as e: print("Drive flush warning:", e)
runtime.unassign()